# Planning

- Many agents use TODO lists as a critical navigation tool for steering through long-running, complex tasks. 
- Agents become vulnerable to drifting off-topic or forgetting earlier objectives during lengthy conversations or complicated tasks. 
- By continuously rewriting and updating the TODO list, agents effectively recite their objectives at the end of the context, helping to stay focused on task and prevent mission drift.

In [ ]:
from collections import Counter
from typing import Annotated, Any, Literal

from langchain.agents import AgentState, create_agent
from langchain.messages import ToolMessage
from langchain.tools import InjectedState, InjectedToolCallId, tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Command
from loguru import logger
from pydantic import BaseModel
from tavily import TavilyClient

from chain_reaction.config import APIKeys, ModelName, get_chat_model
from chain_reaction.utils import format_messages

## Deep Agent State
- Add a list of TODOs for agent to track progress of todo list

In [ ]:
type Status = Literal["pending", "in-progress", "completed"]
STATUS_EMOJI = {"pending": "⏳", "in-progress": "🔄", "completed": "✅"}


class Task(BaseModel):
    """Task item for tracking progress through complex workflows.

    Attributes:
        title (str): Task title.
        description (str): Detailed description of task.
        status (Status): Current status of task. Defaults to "pending".
    """

    title: str
    description: str
    status: Status = "pending"

    def __str__(self) -> str:
        """Short string representation of task."""
        return f"{STATUS_EMOJI.get(self.status, '❓')} {self.title}"


class DeepAgentState(AgentState):
    """Extended agent state that includes task tracking."""

    todos: list[Task]

## TODO list tools

In [ ]:
@tool
def manage_todo_list(
    todos: list[Task] | None,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Create and manage structured task lists for tracking progress through complex workflows.

    Args:
        todos (list[Task] | None): ToDo task list.
        tool_call_id (str): Tool call identifier for message response

    Returns:
        Command: Command to update agent state with new TODO list

    ## When to Use
    - Multi-step or non-trivial tasks requiring coordination
    - When user provides multiple tasks or explicitly requests todo list
    - Avoid for single, trivial actions unless directed otherwise

    ## Structure
    - Maintain one list containing multiple task objects (title, description, status)
    - Use clear, actionable task descriptions
    - Status must be: pending, in_progress, or completed

    ## Best Practices
    - Only one in_progress task at a time
    - Mark completed immediately when task is fully done
    - Always send the full updated list when making changes
    - Prune irrelevant items to keep list focused

    ## Progress Updates
    - Call this function again to change task status or edit content
    - Reflect real-time progress; don't batch completions
    - If blocked, keep blocked task in_progress and add new task describing blocker
    """
    task_counts = Counter(task.status for task in todos or [])
    logger.info("updating todo list: {task_counts}", task_counts=task_counts)
    return Command(
        update={
            "todos": todos,
            "messages": [ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)],
        }
    )


@tool
def read_todo_list(
    state: Annotated[DeepAgentState, InjectedState],
) -> str:
    """Read the current ToDo task list from agent state.

    Args:
        state (Annotated[DeepAgentState, InjectedState]): Injected agent state containing the current TODO list

    Returns:
        str: Formatted string representation of the current TODO list
    """
    todos: list[Task] | None = state.get("todos")
    logger.info("reading {num_tasks} tasks from todo list", num_tasks=len(todos or []))
    if not todos:
        return "No tasks in todos list"
    return "\n".join(f"{i}. {task!s} ({task.status}) - {task.description}" for i, task in enumerate(todos))


todo_tools = [manage_todo_list, read_todo_list]
TODO_USAGE_INSTRUCTIONS = f"""Based upon the user's request:
1. Use the {manage_todo_list.func.__name__} tool to create TODO at the start of a user request, per the tool description.
2. After you accomplish a TODO, use the {read_todo_list.func.__name__} to read the TODOs in order to remind yourself of the plan.
3. Reflect on what you've done and the TODO.
4. Mark you task as completed, and proceed to the next TODO.
5. Continue this process until you have completed all TODOs.

IMPORTANT: Always create a research plan of TODOs and conduct research following the above guidelines for ANY user request.
"""  # noqa: E501

In [ ]:
print(TODO_USAGE_INSTRUCTIONS)

## Web search tool

In [ ]:
tavily_client = TavilyClient(api_key=APIKeys().tavily.get_secret_value())


@tool
def search_web(query: str) -> dict[str, Any]:
    """Performs a web search using Tavily.

    Args:
        query (str): The search query.

    Returns:
        dict[str, Any]: The search results.
    """
    logger.info("Running web query: {query}", query=query)
    return tavily_client.search(query=query)

In [ ]:
query = "Model Context Protocol (MCP) use cases"
search_result = search_web.invoke(query)

## Summarization tool

In [ ]:
summarization_model = get_chat_model(model_name=ModelName.CLAUDE_SONNET)
summarization_prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert at summarizing topic search results into factual, cohesive summaries. Stick only to the facts and content provided to you. NEVER make things up.",  # noqa: E501
    ),
    (
        "human",
        "Provide a {summary_type} summary of the following results for the topic query: {topic_query}\nTopic Results: {topic_results}",  # noqa: E501
    ),
])
summarization_chain = summarization_prompt_template | summarization_model | StrOutputParser()


@tool
def summarize_topic(
    topic_query: str, topic_results: list[str], *, summary_type: Literal["short", "detailed"] = "short"
) -> str:
    """Summarize the content of search results for a topic.

    Args:
        topic_query (str): Query used to generate topic results.
        topic_results (list[str]): List of content from topic search.
        summary_type (Literal["short", "detailed"]): Style of summary. Defaults to "short"

    Returns:
        str: Topic summary from results.
    """
    logger.info(
        "Generating {summary_type} summary using {num_results} search results",
        summary_type=summary_type,
        num_results=len(topic_results),
    )
    return summarization_chain.invoke({
        "summary_type": summary_type,
        "topic_query": topic_query,
        "topic_results": "\n".join(f"{i} {content}" for i, content in enumerate(topic_results)),
    })

In [ ]:
summary = summarize_topic.invoke({
    "topic_query": query,
    "topic_results": [r.get("content", "") for r in search_result.get("results", [])],
    "summary_type": "detailed",
})

print(summary)

## Create search & summarize agent

In [ ]:
system_prompt = f"""
You are a helpful search and summarization assistant.

Use your search and summarization tools to answer a users question. It's your job
to determine whether the user needs a detailed or short summary based on their query.

{TODO_USAGE_INSTRUCTIONS}
"""
print(system_prompt)

In [ ]:
agent = create_agent(
    model=get_chat_model(),
    tools=[search_web, summarize_topic, *todo_tools],
    system_prompt=system_prompt,
    state_schema=DeepAgentState,
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me a comprehensive summary of the Model Context Protocol (MCP) using FastMCP.",
        }
    ],
    "todos": [],
})

format_messages(result["messages"])